# 17. Validação e caminhos de erro

Aqui confiro se uma entrada errada levanta mesmo uma exceção nos módulos do app (requisitos F1, F4, F5, F6 e F10).

In [1]:
import sys, os
_cwd = os.getcwd()
RAIZ = os.path.dirname(_cwd) if os.path.basename(_cwd) == 'tests' else _cwd
if RAIZ not in sys.path:
    sys.path.insert(0, RAIZ)
import numpy as np, pandas as pd
from app import dal, nucleo
from app.agente import Investidor
from app.mercado import RendaVariavel
from app.principal import executar_pipeline


def deve_levantar(exc, fn, *a, **k):
    """Diz se chamar fn(*a, **k) levanta a excecao exc."""
    try:
        fn(*a, **k)
    except exc:
        return True
    return False

Teste: as validações do Investidor, do mercado e do dal.

In [2]:
assert deve_levantar(ValueError, Investidor, -1.0, 0.96, 1.0, 30)
assert deve_levantar(ValueError, Investidor, 5.0, 1.5, 1.0, 30)
assert deve_levantar(RuntimeError, Investidor(5.0,0.96,1.0,12).fracoes_consumo)
assert deve_levantar(ValueError, RendaVariavel, pd.DataFrame({'data':['2000-01','2000-02'],'ibov':[0.01,np.nan]}))
assert deve_levantar(KeyError, dal.calcular_retornos, pd.DataFrame({'ibov':[1.,2.,3.]}))
assert deve_levantar(ValueError, dal.gravar_sqlite, pd.DataFrame({'a':[1]}), 'x.db', 'tab invalida!')

In [3]:
# horizonte e periodos_por_ano nao tem valor padrao: 60 periodos sao 5 anos no
# mensal e uns 3 meses no diario
dados = {'retornos': pd.DataFrame({'data':['2020-01','2020-02','2020-03'],
                                   'ibov':[0.01,-0.02,0.03],'cdi':[0.005]*3}),
         'ativos':['ibov'], 'n_scenarios':1000, 'n_paths':10}
cfg = {**dados, 'periodos_por_ano': 12}
assert deve_levantar(ValueError, executar_pipeline, cfg)  # sem 'horizonte'
assert executar_pipeline({**cfg, 'horizonte': 4})['horizonte'] == 4
assert deve_levantar(ValueError, executar_pipeline, {**dados, 'horizonte': 4})  # sem 'periodos_por_ano'
assert deve_levantar(ValueError, executar_pipeline, {**cfg, 'horizonte': 4, 'periodos_por_ano': 0})

In [4]:
# 'beta' (por periodo) e 'beta_anual' juntos sao ambiguos
assert deve_levantar(ValueError, executar_pipeline, {**cfg, 'horizonte': 4,
                                                     'beta': 0.9, 'beta_anual': 0.9})

In [5]:
# frequencia desconhecida na DAL
assert deve_levantar(ValueError, dal._formato_data, '1wk')
assert deve_levantar(ValueError, dal.baixar_cdi_bcb, '2020-01-01', '2020-02-01', 'mensal')

In [6]:
# sem alpha* finito: amostra so com R > rf (arbitragem), entao a FOC nao troca de sinal
R_arb = (1 + np.abs(np.random.default_rng(0).normal(0.05, 0.01, 20_000))).reshape(-1, 1)

assert deve_levantar(RuntimeError, nucleo.resolver_alpha_otimo, R_arb, 1.0005, 4.0)